# CNBC news fetcher (last 5 years)
Stage 1 crawls CNBC's daily sitemap for article URLs. Stage 2 downloads each article's text.
Both stages are resumable: if Colab disconnects, just re-run the cells.

In [ ]:
!pip -q install requests beautifulsoup4 lxml pandas tqdm

import re, os, json, time, calendar, requests, pandas as pd
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from google.colab import drive

drive.mount("/content/drive")

# ---------- config ----------
OUT = "/content/drive/MyDrive/nlp_project/cnbc"
END = date.today()
START = END.replace(year=END.year - 5)      # last five years
WORKERS = 6
HEADERS = {"User-Agent": "Mozilla/5.0 (university NLP project)"}
# ----------------------------

os.makedirs(f"{OUT}/urls_by_month", exist_ok=True)
print("Range:", START, "->", END)

Mounted at /content/drive
Range: 2021-09-28 -> 2026-09-28


In [ ]:
def day_url(d):
    return f"https://www.cnbc.com/site-map/articles/{d.year}/{calendar.month_name[d.month]}/{d.day}/"

def fetch_day(d, retries=3):
    """List of {date, url, title} for one day; None if every retry failed.
    Article paths sit in script JSON with '/' escaped as \\u002F; keeping only
    paths dated exactly d drops menu/nav noise."""
    pat = re.compile(re.escape(d.strftime("/%Y/%m/%d/")) + r"[a-z0-9\-]+\.html")
    for i in range(retries):
        try:
            r = requests.get(day_url(d), headers=HEADERS, timeout=20)
            if r.status_code == 200:
                paths = sorted(set(pat.findall(r.text.replace("\\u002F", "/"))))
                return [{"date": d.isoformat(),
                         "url": "https://www.cnbc.com" + p,
                         "title": p.rsplit("/", 1)[1][:-5].replace("-", " ")} for p in paths]
            if r.status_code == 404:
                return []
        except requests.RequestException:
            pass
        time.sleep(2 ** i)
    return None

def body_from_script(html):
    """Fallback: body lives in page script JSON as "articleBodyText"."""
    m = re.search(r'"articleBodyText":"((?:[^"\\]|\\.)*)"', html)
    if not m:
        return ""
    try:
        txt = json.loads('"' + m.group(1) + '"')
    except Exception:
        return ""
    return BeautifulSoup(txt, "lxml").get_text(" ", strip=True)

def parse_article(html):
    soup = BeautifulSoup(html, "lxml")
    head, ptime, body = None, None, ""
    for s in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(s.string or "")
        except Exception:
            continue
        for item in (data if isinstance(data, list) else [data]):
            if isinstance(item, dict) and item.get("@type") in ("NewsArticle", "Article"):
                head = head or item.get("headline")
                ptime = ptime or item.get("datePublished")
                body = body or item.get("articleBody", "")
    if not ptime:
        m = soup.find("meta", property="article:published_time")
        ptime = m.get("content") if m else None
    if not head:
        m = soup.find("meta", property="og:title")
        head = m.get("content") if m else None
    if len(body) < 200:
        ps = soup.select(".ArticleBody-articleBody p") or soup.select("div.group p")
        body = " ".join(p.get_text(" ", strip=True) for p in ps)
    if len(body) < 200:
        body = body_from_script(html) or body
    return head, ptime, body

def fetch_article(row, retries=3):
    url, d = row
    empty = {"date": d, "url": url, "headline": None, "published_time": None,
             "body": "", "n_chars": 0}
    for i in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=20)
            if r.status_code == 200:
                h, t, b = parse_article(r.text)
                return {**empty, "headline": h, "published_time": t,
                        "body": b, "n_chars": len(b), "status": 200}
            if r.status_code in (404, 410):
                return {**empty, "status": r.status_code}
        except requests.RequestException:
            pass
        time.sleep(2 ** i)
    return {**empty, "status": -1}      # failed: not saved, retried on next run

# quick sanity check
t = fetch_day(date(2025, 1, 15))
print(len(t), t[:2])

80 [{'date': '2025-01-15', 'url': 'https://www.cnbc.com/2025/01/15/10-things-to-watch-in-the-stock-market-wednesday-the-bank-earnings-edition.html', 'title': '10 things to watch in the stock market wednesday the bank earnings edition'}, {'date': '2025-01-15', 'url': 'https://www.cnbc.com/2025/01/15/5-things-to-know-before-the-stock-market-opens-wednesday-january-15.html', 'title': '5 things to know before the stock market opens wednesday january 15'}]


## Stage 1 — article URLs (one checkpoint file per month)

In [ ]:
months, d = [], date(START.year, START.month, 1)
while d <= END:
    months.append(d)
    d = date(d.year + (d.month == 12), d.month % 12 + 1, 1)

for m in months:
    path = f"{OUT}/urls_by_month/{m:%Y-%m}.csv"
    if os.path.exists(path):
        continue
    last = calendar.monthrange(m.year, m.month)[1]
    s, e = max(START, m), min(END, date(m.year, m.month, last))
    days = [s + timedelta(n) for n in range((e - s).days + 1)]

    rows, failed = [], []
    with ThreadPoolExecutor(WORKERS) as ex:
        futs = {ex.submit(fetch_day, x): x for x in days}
        for f in tqdm(as_completed(futs), total=len(days), desc=f"{m:%Y-%m}"):
            out = f.result()
            (failed.append(futs[f]) if out is None else rows.extend(out))
    for x in failed[:]:                       # one sequential retry
        out = fetch_day(x)
        if out is not None:
            rows.extend(out); failed.remove(x)
    if failed:
        print("  skipped (will not checkpoint, re-run to retry):", len(failed), "days")
        continue
    pd.DataFrame(rows, columns=["date", "url", "title"]).to_csv(path, index=False)

urls = (pd.concat([pd.read_csv(f"{OUT}/urls_by_month/{f}")
                   for f in sorted(os.listdir(f"{OUT}/urls_by_month"))])
          .drop_duplicates("url").sort_values("date"))
urls.to_csv(f"{OUT}/cnbc_urls.csv", index=False)
print(len(urls), "URLs |", urls.date.nunique(), "days")

2021-09:   0%|          | 0/3 [00:00<?, ?it/s]

2021-10:   0%|          | 0/31 [00:00<?, ?it/s]

2021-11:   0%|          | 0/30 [00:00<?, ?it/s]

2021-12:   0%|          | 0/31 [00:00<?, ?it/s]

2022-01:   0%|          | 0/31 [00:00<?, ?it/s]

...

2026-06:   0%|          | 0/30 [00:00<?, ?it/s]

2026-07:   0%|          | 0/31 [00:00<?, ?it/s]

2026-08:   0%|          | 0/31 [00:00<?, ?it/s]

2026-09:   0%|          | 0/28 [00:00<?, ?it/s]

97395 URLs | 1827 days


## Stage 2 — article text (resumable, appends in chunks)

In [ ]:
body_path = f"{OUT}/cnbc_bodies.csv"
done = set(pd.read_csv(body_path)["url"]) if os.path.exists(body_path) else set()
todo = [r for r in zip(urls["url"], urls["date"]) if r[0] not in done]
print(len(done), "done |", len(todo), "to fetch")

CHUNK = 300
for i in range(0, len(todo), CHUNK):
    chunk = todo[i:i + CHUNK]
    with ThreadPoolExecutor(WORKERS) as ex:
        out = list(tqdm(ex.map(fetch_article, chunk), total=len(chunk),
                        desc=f"{i}/{len(todo)}"))
    ok = pd.DataFrame([o for o in out if o["status"] != -1])
    if len(ok):
        ok.to_csv(body_path, mode="a", index=False, header=not os.path.exists(body_path))

0 done | 97395 to fetch


0/97395:   0%|          | 0/300 [00:00<?, ?it/s]

300/97395:   0%|          | 0/300 [00:00<?, ?it/s]

600/97395:   0%|          | 0/300 [00:00<?, ?it/s]

900/97395:   0%|          | 0/300 [00:00<?, ?it/s]

...

96000/97395:   0%|          | 0/300 [00:00<?, ?it/s]

96300/97395:   0%|          | 0/300 [00:00<?, ?it/s]

96600/97395:   0%|          | 0/300 [00:00<?, ?it/s]

96900/97395:   0%|          | 0/300 [00:00<?, ?it/s]

97200/97395:   0%|          | 0/195 [00:00<?, ?it/s]

## Summary

In [ ]:
import pandas as pd
OUT = "/content/drive/MyDrive/nlp_project/cnbc"
body_path = f"{OUT}/cnbc_bodies.csv"
res = pd.read_csv(body_path)
print(len(res), "articles saved")
print("status:", res.status.value_counts().to_dict())
print("short (<300 chars, likely paywalled):", (res.n_chars < 300).sum())
res.head()

97385 articles saved
status: {200: 96964, 404: 421}
short (<300 chars, likely paywalled): 665


,date,url,headline,published_time,body,n_chars,status
0,2021-09-28,https://www.cnbc.com/2021/09/28/unvaccinated-a...,Unvaccinated Americans falsely say the need fo...,2021-09-28T09:00:01+0000,The divide in attitudes on Covid-19 vaccines b...,5687,200
1,2021-09-28,https://www.cnbc.com/2021/09/28/deal-for-state...,Deal for state and local tax deduction cap may...,2021-09-28T17:49:59+0000,"Rep. Tom Suozzi, D-N.Y., is expecting an agree...",2832,200
2,2021-09-28,https://www.cnbc.com/2021/09/28/defendants-in-...,Two defendants in shell company stock hijackin...,2021-09-28T19:49:26+0000,Two of the three men criminally charged with a...,5071,200
3,2021-09-28,https://www.cnbc.com/2021/09/28/defense-secret...,Defense Secretary Austin admits in Senate test...,2021-09-28T20:47:49+0000,WASHINGTON – The Pentagon acknowledged Tuesday...,4814,200
4,2021-09-28,https://www.cnbc.com/2021/09/28/ecosperity-wee...,Trading carbon credits alone will not solve th...,2021-09-28T07:26:20+0000,"In a global effort to fight climate change, ma...",2507,200
